# ND Week 6 — Decision Trees and Random Forests
**DX799 | Cindy Lopez**

**Question:** How well can these models predict customer conversion and campaign conversion counts? We use three datasets independently. The customer target is binary `Conversion`; the campaign targets are observed numerical `Conversions` counts, so we use regression for those two datasets. Metrics across different targets are not directly comparable.

## 1. Data loading, EDA and leakage safeguards
The uncleaned Google Ads file requires currency cleaning and missing-data treatment. Imputation, encoding and scaling are fitted only on training folds. We exclude identifiers, conversion-rate proxies, revenue and ROI from predictors to avoid outcome leakage.

In [1]:
import numpy as np, pandas as pd, os, warnings
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from statsmodels.stats.outliers_influence import variance_inflation_factor
from IPython.display import display

FILES = {
 'Customer conversion': 'digital_marketing_campaign_dataset.csv',
 'Google Ads': 'GoogleAds_DataAnalytics_Sales_Uncleaned.csv',
 'Product performance': 'marketing_and_product_performance.csv'
}
for label, filename in FILES.items():
    if not os.path.isfile(filename):
        raise FileNotFoundError(f'Missing {filename}. Place all three CSVs in the notebook folder or upload them to Colab.')
print('All three CSV files found.')
data={name:pd.read_csv(filename) for name,filename in FILES.items()}
for name, df in data.items():
    print('\n',name,'shape:',df.shape,'missing cells:',int(df.isna().sum().sum()),'duplicates:',int(df.duplicated().sum()))
    display(df.head(2))
    display(df.isna().sum().sort_values(ascending=False).head(5).to_frame('missing'))
ads=data['Google Ads'].copy()
for col in ['Cost','Sale_Amount']:
    ads[col]=pd.to_numeric(ads[col].astype(str).str.replace(r'[$,]','',regex=True),errors='coerce')
for col in ['Clicks','Impressions','Leads','Conversions','Conversion Rate']:
    ads[col]=pd.to_numeric(ads[col],errors='coerce')
# We deliberately exclude Leads, Conversion Rate and Sale_Amount as outcome-adjacent or downstream variables.
config={
 'Customer conversion':{'frame':data['Customer conversion'],'target':'ConversionRate','numeric':['Age','Income','AdSpend','ClickThroughRate','WebsiteVisits','PagesPerVisit','TimeOnSite','SocialShares','EmailOpens','EmailClicks','PreviousPurchases','LoyaltyPoints'],'categorical':['Gender','CampaignChannel','CampaignType'],'poly':['AdSpend','ClickThroughRate','TimeOnSite']},
 'Google Ads':{'frame':ads,'target':'Conversions','numeric':['Clicks','Impressions','Cost'],'categorical':['Campaign_Name','Location','Device'],'poly':['Clicks','Impressions','Cost']},
 'Product performance':{'frame':data['Product performance'],'target':'Conversions','numeric':['Budget','Clicks','Discount_Level','Bundle_Price'],'categorical':['Subscription_Tier','Common_Keywords'],'poly':['Budget','Clicks','Discount_Level']}
}
for name,c in config.items():
    print(name,'target:',c['target'],'range:',c['frame'][c['target']].min(),c['frame'][c['target']].max())

All three CSV files found.

 Customer conversion shape: (8000, 20) missing cells: 0 duplicates: 0


,CustomerID,Age,Gender,Income,CampaignChannel,CampaignType,AdSpend,ClickThroughRate,ConversionRate,WebsiteVisits,PagesPerVisit,TimeOnSite,SocialShares,EmailOpens,EmailClicks,PreviousPurchases,LoyaltyPoints,AdvertisingPlatform,AdvertisingTool,Conversion
0,8000,56,Female,136912,Social Media,Awareness,6497.870068,0.043919,0.088031,0,2.399017,7.396803,19,6,9,4,688,IsConfid,ToolConfid,1
1,8001,69,Male,41760,Email,Retention,3898.668606,0.155725,0.182725,42,2.917138,5.352549,5,2,7,2,3459,IsConfid,ToolConfid,1


,missing
CustomerID,0
Age,0
Gender,0
Income,0
CampaignChannel,0



 Google Ads shape: (2600, 13) missing cells: 1150 duplicates: 0


,Ad_ID,Campaign_Name,Clicks,Impressions,Cost,Leads,Conversions,Conversion Rate,Sale_Amount,Ad_Date,Location,Device,Keyword
0,A1000,DataAnalyticsCourse,104.0,4498.0,$231.88,14.0,7.0,0.058,$1892,2024-11-16,hyderabad,desktop,learn data analytics
1,A1001,DataAnalyticsCourse,173.0,5107.0,$216.84,10.0,8.0,0.046,$1679,20-11-2024,hyderabad,mobile,data analytics course


,missing
Conversion Rate,626
Sale_Amount,139
Clicks,112
Cost,97
Conversions,74



 Product performance shape: (10000, 17) missing cells: 0 duplicates: 0


,Campaign_ID,Product_ID,Budget,Clicks,Conversions,Revenue_Generated,ROI,Customer_ID,Subscription_Tier,Subscription_Length,Flash_Sale_ID,Discount_Level,Units_Sold,Bundle_ID,Bundle_Price,Customer_Satisfaction_Post_Refund,Common_Keywords
0,CMP_RLSDVN,PROD_HBJFA3,41770.45,4946,73,15520.09,1.94,CUST_1K7G39,Premium,4,FLASH_1VFK5K,43,34,BNDL_29U6W5,433.80,4,Affordable
1,CMP_JHHUE9,PROD_OE8YNJ,29900.93,570,510,30866.17,0.76,CUST_0DWS6F,Premium,4,FLASH_1M6COK,28,97,BNDL_ULV60J,289.29,2,Innovative


,missing
Campaign_ID,0
Product_ID,0
Budget,0
Clicks,0
Conversions,0


Customer conversion target: ConversionRate range: 0.0100177828330717 0.1999947079999166
Google Ads target: Conversions range: 3.0 10.0
Product performance target: Conversions range: 1 999


In [2]:
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, KFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, r2_score, mean_squared_error, mean_absolute_error, confusion_matrix
from IPython.display import display
import matplotlib.pyplot as plt

def ohe():
    try: return OneHotEncoder(handle_unknown='ignore',sparse_output=False)
    except TypeError: return OneHotEncoder(handle_unknown='ignore',sparse=False)
def prep(c,scale=True):
    ns=[('impute',SimpleImputer(strategy='median'))]
    if scale: ns.append(('scale',StandardScaler()))
    return ColumnTransformer([('numeric',Pipeline(ns),c['numeric']),('categorical',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('onehot',ohe())]),c['categorical'])])

for name,c in config.items():
    target='Conversion' if name=='Customer conversion' else 'Conversions'
    y=pd.to_numeric(c['frame'][target],errors='coerce')
    print(name,'rows:',len(c['frame']),'target missing:',int(y.isna().sum()),'target median:',y.median(),'target std:',y.std())

Customer conversion rows: 8000 target missing: 0 target median: 1.0 target std: 0.3290308232678703
Google Ads rows: 2600 target missing: 74 target median: 7.0 target std: 2.2725732687366835
Product performance rows: 10000 target missing: 0 target median: 499.0 target std: 289.47949475584346


## 2. Decision trees versus random forests
A decision tree creates interpretable recursive partitions but can overfit. We constrain tree depth and minimum leaf size. A random forest averages predictions across bootstrapped trees and randomly sampled features to reduce variance. For the customer-level binary outcome we use classifiers; for campaign conversion counts we use regressors. All comparisons use an untouched 20% test set.

In [3]:
from sklearn.tree import DecisionTreeClassifier,DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier,RandomForestRegressor
rows=[];details={}
for name,c in config.items():
    target='Conversion' if name=='Customer conversion' else 'Conversions'
    df=c['frame'].copy(); yy=pd.to_numeric(df[target],errors='coerce'); ok=yy.notna(); df=df.loc[ok]; yy=yy.loc[ok]
    cls=(name=='Customer conversion'); y=yy.astype(int) if cls else yy.astype(float)
    X=df[c['numeric']+c['categorical']]
    Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42,stratify=y if cls else None)
    for model_name in ['Decision tree','Random forest']:
        if cls:
            model=DecisionTreeClassifier(max_depth=5,min_samples_leaf=20,random_state=42) if model_name=='Decision tree' else RandomForestClassifier(n_estimators=120,max_depth=10,min_samples_leaf=5,random_state=42,n_jobs=-1)
        else:
            model=DecisionTreeRegressor(max_depth=5,min_samples_leaf=20,random_state=42) if model_name=='Decision tree' else RandomForestRegressor(n_estimators=120,max_depth=10,min_samples_leaf=5,random_state=42,n_jobs=-1)
        pipe=Pipeline([('prep',prep(c,scale=False)),('model',model)])
        pipe.fit(Xtr,ytr);p=pipe.predict(Xte)
        if cls:
            prob=pipe.predict_proba(Xte)[:,1]
            row={'dataset':name,'model':model_name,'train_accuracy':accuracy_score(ytr,pipe.predict(Xtr)),'test_accuracy':accuracy_score(yte,p),'balanced_accuracy':balanced_accuracy_score(yte,p),'F1_macro':f1_score(yte,p,average='macro'),'ROC_AUC':roc_auc_score(yte,prob),'minority_recall':((p[yte==0]==0).mean() if (yte==0).any() else np.nan)}
        else:
            row={'dataset':name,'model':model_name,'train_R2':r2_score(ytr,pipe.predict(Xtr)),'test_R2':r2_score(yte,p),'RMSE':np.sqrt(mean_squared_error(yte,p)),'MAE':mean_absolute_error(yte,p)}
        rows.append(row);details[(name,model_name)]=(Xtr,Xte,ytr,yte,p,pipe)
        print(name,model_name,'completed')
results=pd.DataFrame(rows);display(results.round(4))

Customer conversion Decision tree completed


Customer conversion Random forest completed
Google Ads Decision tree completed


Google Ads Random forest completed
Product performance Decision tree completed


Product performance Random forest completed


,dataset,model,train_accuracy,test_accuracy,balanced_accuracy,F1_macro,ROC_AUC,minority_recall,train_R2,test_R2,RMSE,MAE
0,Customer conversion,Decision tree,0.8939,0.8800,0.5629,0.5804,0.6921,0.1414,NaN,NaN,NaN,NaN
1,Customer conversion,Random forest,0.9188,0.8838,0.5390,0.5422,0.7933,0.0808,NaN,NaN,NaN,NaN
2,Google Ads,Decision tree,NaN,NaN,NaN,NaN,NaN,NaN,0.0187,-0.0257,2.2843,1.9750
3,Google Ads,Random forest,NaN,NaN,NaN,NaN,NaN,NaN,0.2526,-0.0299,2.2889,1.9740
4,Product performance,Decision tree,NaN,NaN,NaN,NaN,NaN,NaN,0.0187,-0.0177,293.5226,254.3638
5,Product performance,Random forest,NaN,NaN,NaN,NaN,NaN,NaN,0.1511,-0.0106,292.4926,253.3012


## 3. Hyperparameter tuning: random forest on customer conversions
We tune tree depth and minimum leaf size with three-fold stratified cross-validation using ROC-AUC. These control complexity: shallower trees and larger leaves reduce overfitting, while overly restrictive values may underfit. We compare CV scores with untouched test performance.

In [4]:
name='Customer conversion';c=config[name]
Xtr,Xte,ytr,yte,_,_=details[(name,'Random forest')]
pipe=Pipeline([('prep',prep(c,scale=False)),('model',RandomForestClassifier(n_estimators=120,random_state=42,n_jobs=-1))])
search=GridSearchCV(pipe,{'model__max_depth':[5,10],'model__min_samples_leaf':[5,20]},scoring='roc_auc',cv=StratifiedKFold(3,shuffle=True,random_state=42),n_jobs=1)
search.fit(Xtr,ytr);p=search.predict(Xte);prob=search.predict_proba(Xte)[:,1]
print('Best parameters:',search.best_params_,'CV ROC-AUC:',round(search.best_score_,4))
print('Test ROC-AUC:',round(roc_auc_score(yte,prob),4),'balanced accuracy:',round(balanced_accuracy_score(yte,p),4))
print('Confusion matrix:',confusion_matrix(yte,p).tolist())

Best parameters: {'model__max_depth': 10, 'model__min_samples_leaf': 5} CV ROC-AUC: 0.8172
Test ROC-AUC: 0.7933 balanced accuracy: 0.539
Confusion matrix: [[16, 182], [4, 1398]]


## 4. Interpretation and limitations
Compare train and test scores to identify overfitting. For imbalanced customer outcomes, overall accuracy can mask poor minority recall. ROC-AUC measures ranking quality and is not equivalent to performance at the 0.5 decision threshold. Campaign conversion counts are not the same outcome as individual customer conversion. This is predictive analysis; feature importance or model association is not causal attribution.

**Sources:** El Kharoua (2024), Nayak (2024), Shah (2024), Kaggle datasets. scikit-learn documentation: https://scikit-learn.org/stable/supervised_learning.html

**AI-use disclosure:** ChatGPT assisted with notebook code, structure and explanation. All quantitative claims should be checked against the saved executed output.

## 5. Deeper analysis: model complexity and class-specific errors

The comparison above gives overall scores, but a stronger evaluation asks **how performance changes as a tree becomes more complex** and **which customers are misclassified**. We use the same held-out customer test set and vary tree depth. We inspect train versus test ROC-AUC rather than relying only on accuracy, because the classes are imbalanced. These diagnostics do not use the test set to select hyperparameters; tuning remains based on cross-validation in Section 3.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, classification_report

# Inspect the held-out predictions from the cross-validated random forest.
print('Tuned random forest — classification report:')
print(classification_report(yte, p, digits=3, zero_division=0))
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(yte, p, ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Tuned random forest: customer conversion confusion matrix')
plt.tight_layout(); plt.show()

# ROC curve compares ranking ability across probability thresholds.
fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(yte, prob, ax=ax, name='Tuned random forest')
ax.plot([0, 1], [0, 1], '--', color='gray', label='Random ranking')
ax.set_title('Tuned random forest: held-out ROC curve')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Diagnostic only: show how model complexity changes training and held-out scores.
# Do not select depth based on this test-set plot; use training CV for selection.
depth_rows = []
for depth in [2, 4, 6, 8, 10, 14, None]:
    candidate = Pipeline([
        ('prep', prep(c, scale=False)),
        ('model', DecisionTreeClassifier(max_depth=depth, min_samples_leaf=5, random_state=42))
    ])
    candidate.fit(Xtr, ytr)
    depth_rows.append({
        'max_depth': str(depth),
        'train_auc': roc_auc_score(ytr, candidate.predict_proba(Xtr)[:, 1]),
        'test_auc': roc_auc_score(yte, candidate.predict_proba(Xte)[:, 1])
    })
depth_results = pd.DataFrame(depth_rows)
display(depth_results.round(4))
ax = depth_results.plot(x='max_depth', y=['train_auc', 'test_auc'], marker='o', figsize=(8, 4))
ax.set(xlabel='Maximum tree depth', ylabel='ROC-AUC', title='Tree complexity diagnostic (not for model selection)')
ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()

### Interpretation prompts for the final report

After running these cells, record the **actual values** before drafting the Milestone One conclusions:

- At which depths does training ROC-AUC rise while test ROC-AUC stagnates or declines? That is evidence of overfitting.
- How many non-converting customers are correctly identified? Why might overall accuracy conceal that result?
- Does the tuned forest improve ROC-AUC relative to the untuned forest? Is that improvement large enough to matter for a marketing decision?
- Why might a different classification threshold be appropriate if the cost of missing a non-converter differs from the cost of a false alarm?

These are diagnostic questions, not conclusions assumed in advance. Do not claim improvements unless the executed outputs support them.